# PacketLens Experiments Notebook

Interactive testing of the NIDS inference pipeline:
- **Experiment A:** White-Box Testing (Local ONNX)
- **Experiment B:** Black-Box Testing (gRPC Server)
- **Experiment C:** Stress Test & Latency Analysis

---
## Setup & Imports

In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path includes project: {str(PROJECT_ROOT) in sys.path}")

Project root: /home/mahmoud/My-SSD-Space/Projects/PacketLens
Python path includes project: True


In [2]:
import json
import time
from typing import Iterator

import grpc
import numpy as np
import onnxruntime as ort

# Import proto stubs
from services.inference.proto import packetlens_pb2, packetlens_pb2_grpc

print("All imports successful!")

All imports successful!


---
## Experiment A: White-Box Testing (Local ONNX)

Load the ONNX model directly and verify it works without the gRPC layer.

In [3]:
# Paths to artifacts
MODEL_PATH = PROJECT_ROOT / "services/inference/model_store/model.onnx"
LABEL_MAPPING_PATH = PROJECT_ROOT / "data/processed/label_mapping.json"

print(f"Model exists: {MODEL_PATH.exists()}")
print(f"Labels exist: {LABEL_MAPPING_PATH.exists()}")

Model exists: True
Labels exist: True


In [4]:
# Load ONNX model
session = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])

# Get input/output info
input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape
output_names = [o.name for o in session.get_outputs()]

print(f"Input name:  {input_name}")
print(f"Input shape: {input_shape}")
print(f"Outputs:     {output_names}")

Input name:  features
Input shape: [None, 54]
Outputs:     ['label', 'probabilities']


In [5]:
# Load label mapping
with open(LABEL_MAPPING_PATH, "r") as f:
    label_data = json.load(f)

int_to_label = {int(k): v for k, v in label_data["int_to_label"].items()}
print(f"Loaded {len(int_to_label)} class labels:")
for idx, label in list(int_to_label.items())[:5]:
    print(f"  {idx}: {label}")
print(f"  ...")

Loaded 33 class labels:
  0: Benign
  1: Botnet
  2: Bruteforce-FTP
  3: Bruteforce-SSH
  4: DDoS
  ...


In [6]:
# Generate random input and run inference
np.random.seed(42)
dummy_input = np.random.rand(1, 54).astype(np.float32)

print(f"Input shape: {dummy_input.shape}")
print(f"Input sample: {dummy_input[0, :5]}... (first 5 features)")

Input shape: (1, 54)
Input sample: [0.37454012 0.9507143  0.7319939  0.5986585  0.15601864]... (first 5 features)


In [7]:
# Run ONNX inference
start = time.perf_counter()
outputs = session.run(None, {input_name: dummy_input})
elapsed_ms = (time.perf_counter() - start) * 1000

# Parse outputs
labels = outputs[0]       # Predicted class indices
probabilities = outputs[1] # Softmax probabilities

pred_idx = int(labels[0])
pred_label = int_to_label.get(pred_idx, f"UNKNOWN_{pred_idx}")
confidence = float(probabilities[0, pred_idx])

print(f"\n=== ONNX Inference Result ===")
print(f"Predicted class: {pred_label}")
print(f"Confidence:      {confidence:.2%}")
print(f"Inference time:  {elapsed_ms:.3f} ms")
print(f"\nTop 5 probabilities:")
top5_idx = np.argsort(probabilities[0])[::-1][:5]
for idx in top5_idx:
    label = int_to_label.get(idx, f"UNKNOWN_{idx}")
    prob = probabilities[0, idx]
    print(f"  {label:25s}: {prob:.4%}")


=== ONNX Inference Result ===
Predicted class: Benign
Confidence:      97.60%
Inference time:  1.161 ms

Top 5 probabilities:
  Benign                   : 97.6001%
  Infiltration             : 2.0792%
  Webattack-SQLi           : 0.0639%
  Webattack-bruteforce     : 0.0279%
  Webattack-XSS            : 0.0277%


---
## Experiment B: Black-Box Testing (gRPC Server)

Test the gRPC inference service running on `localhost:50051`.

> **Note:** Make sure the server is running:
> ```bash
> python -m services.inference.main
> ```

In [8]:
# gRPC connection settings
GRPC_HOST = "localhost"
GRPC_PORT = 50051
GRPC_TARGET = f"{GRPC_HOST}:{GRPC_PORT}"

print(f"Target: {GRPC_TARGET}")

Target: localhost:50051


In [9]:
# Create gRPC channel and stub
channel = grpc.insecure_channel(GRPC_TARGET)
stub = packetlens_pb2_grpc.InferenceServiceStub(channel)

# Check connectivity
try:
    grpc.channel_ready_future(channel).result(timeout=3)
    print(f"Connected to {GRPC_TARGET}")
except grpc.FutureTimeoutError:
    print(f"ERROR: Cannot connect to {GRPC_TARGET}")
    print("Make sure the server is running: python -m services.inference.main")

Connected to localhost:50051


In [10]:
def single_request_stream(flow_id: str, features: np.ndarray) -> Iterator[packetlens_pb2.FeatureVector]:
    """Yield a single request (simulates unary over stream)."""
    yield packetlens_pb2.FeatureVector(
        flow_id=flow_id,
        features=features.tolist(),
    )

# Send a single request
test_features = np.random.rand(54).astype(np.float32)

start = time.perf_counter()
response_stream = stub.Classify(single_request_stream("notebook_test_001", test_features))

for verdict in response_stream:
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"\n=== gRPC Inference Result ===")
    print(f"Flow ID:         {verdict.flow_id}")
    print(f"Predicted class: {verdict.label}")
    print(f"Confidence:      {verdict.confidence:.2%}")
    print(f"Server latency:  {verdict.inference_time_us} us")
    print(f"Round-trip time: {elapsed_ms:.2f} ms")


=== gRPC Inference Result ===
Flow ID:         notebook_test_001
Predicted class: Benign
Confidence:      97.43%
Server latency:  1688 us
Round-trip time: 7.79 ms


---
## Experiment C: Stress Test & Latency Analysis

Send multiple requests and analyze latency distribution.

In [11]:
def batch_request_stream(count: int, n_features: int = 54) -> Iterator[packetlens_pb2.FeatureVector]:
    """Generate a stream of random feature vectors."""
    for i in range(count):
        features = np.random.rand(n_features).astype(np.float32)
        yield packetlens_pb2.FeatureVector(
            flow_id=f"stress_test_{i:05d}",
            features=features.tolist(),
        )

# Run stress test
N_REQUESTS = 100
latencies = []
labels_seen = {}

print(f"Sending {N_REQUESTS} requests...")
start_total = time.perf_counter()

response_stream = stub.Classify(batch_request_stream(N_REQUESTS))

for verdict in response_stream:
    latencies.append(verdict.inference_time_us / 1000)  # Convert to ms
    labels_seen[verdict.label] = labels_seen.get(verdict.label, 0) + 1

total_time = time.perf_counter() - start_total

print(f"Completed in {total_time:.2f}s")

Sending 100 requests...


Completed in 0.24s


In [12]:
# Calculate statistics
latencies_arr = np.array(latencies)

print(f"\n=== Latency Statistics (N={N_REQUESTS}) ===")
print(f"Mean:   {np.mean(latencies_arr):.3f} ms")
print(f"Median: {np.median(latencies_arr):.3f} ms")
print(f"Std:    {np.std(latencies_arr):.3f} ms")
print(f"Min:    {np.min(latencies_arr):.3f} ms")
print(f"Max:    {np.max(latencies_arr):.3f} ms")
print(f"P95:    {np.percentile(latencies_arr, 95):.3f} ms")
print(f"P99:    {np.percentile(latencies_arr, 99):.3f} ms")

print(f"\nThroughput: {N_REQUESTS / total_time:.1f} req/s")

print(f"\n=== Label Distribution ===")
for label, count in sorted(labels_seen.items(), key=lambda x: -x[1]):
    pct = count / N_REQUESTS * 100
    print(f"{label:25s}: {count:4d} ({pct:5.1f}%)")


=== Latency Statistics (N=100) ===
Mean:   1.170 ms
Median: 0.906 ms
Std:    1.143 ms
Min:    0.590 ms
Max:    10.112 ms
P95:    2.352 ms
P99:    6.788 ms

Throughput: 413.2 req/s

=== Label Distribution ===
Benign                   :  100 (100.0%)


In [13]:
# Simple ASCII histogram of latencies
def ascii_histogram(data: np.ndarray, bins: int = 10, width: int = 40) -> None:
    """Print a simple ASCII histogram."""
    counts, edges = np.histogram(data, bins=bins)
    max_count = max(counts)
    
    print(f"\nLatency Distribution (ms):")
    print("-" * 60)
    
    for i, count in enumerate(counts):
        bar_len = int(count / max_count * width) if max_count > 0 else 0
        bar = "#" * bar_len
        label = f"{edges[i]:.2f}-{edges[i+1]:.2f}"
        print(f"{label:15s} | {bar:{width}s} | {count}")

ascii_histogram(latencies_arr)


Latency Distribution (ms):
------------------------------------------------------------
0.59-1.54       | ######################################## | 87
1.54-2.49       | #####                                    | 11
2.49-3.45       |                                          | 0
3.45-4.40       |                                          | 0
4.40-5.35       |                                          | 0
5.35-6.30       |                                          | 0
6.30-7.26       |                                          | 1
7.26-8.21       |                                          | 0
8.21-9.16       |                                          | 0
9.16-10.11      |                                          | 1


In [14]:
# Cleanup
channel.close()
print("gRPC channel closed.")

gRPC channel closed.
